In [1]:
import os
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import timm
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [2]:
# ─────────────────────────────────────────────
#  CONFIGURATION  ← edit these paths
# ─────────────────────────────────────────────
ROOT            = r"D:\ML\env\notebook\data"
METADATA_CSV    = os.path.join(ROOT, "train_metadata.csv")
FEATURES_DIR    = os.path.join(ROOT, "train_features")
LABELS_DIR      = os.path.join(ROOT, "train_labels")

UNET_PATH       = r"pytorch_unet_cloud_30epochfix_256img.pth"
DEEPLAB_PATH    = r"final1_model3.pth"

BANDS           = ["B02", "B03", "B04", "B08"]
IMG_SIZE        = 256
BATCH_SIZE      = 4
THRESHOLD       = 0.5
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ensemble weight: final_prob = UNET_W * unet_prob + DEEPLAB_W * deeplab_prob
UNET_W          = 0.5
DEEPLAB_W       = 0.5

SAVE_PLOTS      = True
PLOT_DIR        = "ensemble_plots"
NUM_VIS         = 8


In [3]:

# ─────────────────────────────────────────────
#  MODEL 1: UNet  (original layer names)
# ─────────────────────────────────────────────
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=1):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.c1   = DoubleConv(in_channels, 16)
        self.c2   = DoubleConv(16, 32)
        self.c3   = DoubleConv(32, 64)
        self.c4   = DoubleConv(64, 128)
        self.bn   = DoubleConv(128, 256)
        self.c5   = DoubleConv(256 + 128, 128)
        self.c6   = DoubleConv(128 + 64,  64)
        self.c7   = DoubleConv(64  + 32,  32)
        self.c8   = DoubleConv(32  + 16,  16)
        self.out  = nn.Sequential(nn.Conv2d(16, out_channels, 1), nn.Sigmoid())

    def forward(self, x):
        c1 = self.c1(x);      p1 = self.pool(c1)
        c2 = self.c2(p1);     p2 = self.pool(c2)
        c3 = self.c3(p2);     p3 = self.pool(c3)
        c4 = self.c4(p3);     p4 = self.pool(c4)
        bn = self.bn(p4)
        u4 = self.up(bn);  u4 = torch.cat([u4, c4], dim=1); c5 = self.c5(u4)
        u3 = self.up(c5);  u3 = torch.cat([u3, c3], dim=1); c6 = self.c6(u3)
        u2 = self.up(c6);  u2 = torch.cat([u2, c2], dim=1); c7 = self.c7(u2)
        u1 = self.up(c7);  u1 = torch.cat([u1, c1], dim=1); c8 = self.c8(u1)
        return self.out(c8)   # already sigmoid → probability [0,1]
# ─────────────────────────────────────────────
#  MODEL 2: ImprovedDeepLabV3+
# ─────────────────────────────────────────────
class XceptionBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = timm.create_model(
            'xception41', pretrained=False, features_only=True, in_chans=4
        )
    def forward(self, x):
        features = self.model(x)
        return features[1], features[3]

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )
    def forward(self, x): return self.conv(x)

class ImprovedDeepLabV3Plus(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = XceptionBackbone()
        self.aspp     = ASPP(1024, 256)
        self.final    = nn.Conv2d(256, 1, 1)

    def forward(self, x):
        h, w = x.shape[-2:]
        _, high = self.backbone(x)
        x = self.aspp(high)
        x = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)
        return torch.sigmoid(self.final(x))   # probability [0,1]


In [4]:
# ─────────────────────────────────────────────
#  DATASETS
#  UNet uses rasterio + TF.resize (no albumentations)
#  DeepLab uses albumentations Normalize
# ─────────────────────────────────────────────
class UNetDataset(Dataset):
    """Matches UNet training pipeline exactly."""
    def __init__(self, chip_ids, features_dir, labels_dir):
        self.chip_ids     = chip_ids
        self.features_dir = features_dir
        self.labels_dir   = labels_dir

    def __len__(self): return len(self.chip_ids)

    def __getitem__(self, idx):
        cid = self.chip_ids[idx]
        bands = []
        for b in BANDS:
            with rasterio.open(os.path.join(self.features_dir, cid, f"{b}.tif")) as s:
                bands.append(s.read(1).astype(np.float32) / 65535.0)
        image = torch.tensor(np.stack(bands, axis=0), dtype=torch.float32)
        image = TF.resize(image, [IMG_SIZE, IMG_SIZE], antialias=True)

        with rasterio.open(os.path.join(self.labels_dir, f"{cid}.tif")) as s:
            label = s.read(1).astype(np.float32)
        label = torch.tensor(label[np.newaxis], dtype=torch.float32)
        label = TF.resize(label, [IMG_SIZE, IMG_SIZE], antialias=True)
        return image, label, cid


class DeepLabDataset(Dataset):
    """Matches DeepLab training pipeline exactly (albumentations Normalize)."""
    MEAN = (0.5, 0.5, 0.5, 0.5)
    STD  = (0.5, 0.5, 0.5, 0.5)

    def __init__(self, chip_ids, features_dir, labels_dir):
        self.chip_ids     = chip_ids
        self.features_dir = features_dir
        self.labels_dir   = labels_dir
        self.transform    = A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Normalize(mean=self.MEAN, std=self.STD, max_pixel_value=65535.0),
        ], additional_targets={'mask': 'mask'})

    def __len__(self): return len(self.chip_ids)

    def __getitem__(self, idx):
        cid = self.chip_ids[idx]
        bands = []
        for b in BANDS:
            with rasterio.open(os.path.join(self.features_dir, cid, f"{b}.tif")) as s:
                bands.append(s.read(1).astype(np.float32))
        image = np.stack(bands, axis=-1)   # (H, W, 4) for albumentations

        with rasterio.open(os.path.join(self.labels_dir, f"{cid}.tif")) as s:
            mask = s.read(1).astype(np.float32)

        aug   = self.transform(image=image, mask=mask)
        image = torch.from_numpy(aug['image']).permute(2, 0, 1).float()
        mask  = torch.from_numpy(aug['mask']).unsqueeze(0).float()
        return image, mask, cid

# ─────────────────────────────────────────────
#  METRICS
# ─────────────────────────────────────────────
def compute_metrics(preds, targets, threshold=0.5):
    p   = (preds > threshold).float()
    eps = 1e-7
    tp  = (p * targets).sum()
    fp  = (p * (1 - targets)).sum()
    fn  = ((1 - p) * targets).sum()
    tn  = ((1 - p) * (1 - targets)).sum()
    return {
        "accuracy":  ((tp + tn) / (tp + tn + fp + fn + eps)).item(),
        "precision": (tp / (tp + fp + eps)).item(),
        "recall":    (tp / (tp + fn + eps)).item(),
        "iou":       (tp / (tp + fp + fn + eps)).item(),
    }


In [6]:

# ─────────────────────────────────────────────
#  VISUALISATION
# ─────────────────────────────────────────────
def save_visualisations(unet_images, labels, ensemble_preds, unet_preds, dl_preds, chip_ids):
    os.makedirs(PLOT_DIR, exist_ok=True)
    n   = min(NUM_VIS, len(unet_images))
    fig = plt.figure(figsize=(20, 4 * n))
    gs  = gridspec.GridSpec(n, 5, figure=fig, hspace=0.4, wspace=0.25)

    for i in range(n):
        img  = unet_images[i]                               # (4, H, W)
        gt   = labels[i, 0].numpy()
        ens  = (ensemble_preds[i, 0].numpy() > THRESHOLD).astype(np.uint8)
        up   = (unet_preds[i, 0].numpy()     > THRESHOLD).astype(np.uint8)
        dp   = (dl_preds[i, 0].numpy()       > THRESHOLD).astype(np.uint8)

        rgb  = np.clip(np.stack([img[2], img[1], img[0]], axis=-1) * 3.5, 0, 1)

        ax0 = fig.add_subplot(gs[i, 0]); ax0.imshow(rgb);              ax0.set_title(f"RGB  {chip_ids[i]}", fontsize=7)
        ax1 = fig.add_subplot(gs[i, 1]); ax1.imshow(gt,  cmap="gray"); ax1.set_title("Ground Truth", fontsize=7)
        ax2 = fig.add_subplot(gs[i, 2]); ax2.imshow(up,  cmap="gray"); ax2.set_title("UNet Pred", fontsize=7)
        ax3 = fig.add_subplot(gs[i, 3]); ax3.imshow(dp,  cmap="gray"); ax3.set_title("DeepLab Pred", fontsize=7)
        ax4 = fig.add_subplot(gs[i, 4]); ax4.imshow(ens, cmap="gray"); ax4.set_title("Ensemble Pred", fontsize=7)
        for ax in [ax0, ax1, ax2, ax3, ax4]: ax.axis("off")

    out_path = os.path.join(PLOT_DIR, "ensemble_predictions.png")
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  Visualisations saved → {out_path}")

# ─────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────
def main():
    print(f"\n{'='*60}")
    print("  Ensemble Evaluation: UNet + DeepLabV3+")
    print(f"{'='*60}")
    print(f"  Device  : {DEVICE}")
    print(f"  Weights : UNet={UNET_W}  DeepLab={DEEPLAB_W}\n")

    # ── Reproduce split ──
    metadata   = pd.read_csv(METADATA_CSV)
    valid_ids  = []
    for cid in metadata["chip_id"].unique():
        feat_ok  = all(os.path.exists(os.path.join(FEATURES_DIR, cid, f"{b}.tif")) for b in BANDS)
        label_ok = os.path.exists(os.path.join(LABELS_DIR, f"{cid}.tif"))
        if feat_ok and label_ok:
            valid_ids.append(cid)

    print(f"  Usable chips : {len(valid_ids)}")
    train_ids, temp   = train_test_split(valid_ids, test_size=0.30, random_state=42)
    val_ids,   test_ids = train_test_split(temp,    test_size=0.667, random_state=42)
    eval_ids = test_ids
    print(f"  Evaluating on TEST set ({len(eval_ids)} chips)\n")

    # ── Dataloaders ──
    unet_loader  = DataLoader(UNetDataset(eval_ids,   FEATURES_DIR, LABELS_DIR), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    dlab_loader  = DataLoader(DeepLabDataset(eval_ids, FEATURES_DIR, LABELS_DIR), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # ── Load models ──
    unet_model = UNet().to(DEVICE)
    unet_model.load_state_dict(torch.load(UNET_PATH, map_location=DEVICE))
    unet_model.eval()
    print("  ✓ UNet loaded")

    dlab_model = ImprovedDeepLabV3Plus().to(DEVICE)
    dlab_model.load_state_dict(torch.load(DEEPLAB_PATH, map_location=DEVICE))
    dlab_model.eval()
    print("  ✓ DeepLabV3+ loaded\n")

    # ── Inference ──
    all_m  = {"accuracy": [], "precision": [], "recall": [], "iou": []}
    unet_m = {"accuracy": [], "precision": [], "recall": [], "iou": []}
    dlab_m = {"accuracy": [], "precision": [], "recall": [], "iou": []}

    vis_unet_imgs, vis_labels, vis_ens, vis_up, vis_dp, vis_ids = [], [], [], [], [], []

    with torch.no_grad():
        for (u_imgs, labels, chip_ids), (d_imgs, _, _) in tqdm(
            zip(unet_loader, dlab_loader), total=len(unet_loader), desc="  Evaluating"
        ):
            u_imgs  = u_imgs.to(DEVICE)
            d_imgs  = d_imgs.to(DEVICE)
            labels  = labels.to(DEVICE)

            u_prob  = unet_model(u_imgs)         # already sigmoid
            d_prob  = dlab_model(d_imgs)         # already sigmoid

            # Calibrated blend: normalise each model to the same scale
            # before weighting so the higher-confidence model doesn't dominate
            u_cal    = (u_prob - u_prob.mean()) / (u_prob.std() + 1e-8)
            d_cal    = (d_prob - d_prob.mean()) / (d_prob.std() + 1e-8)
            ens_prob = torch.sigmoid(UNET_W * u_cal + DEEPLAB_W * d_cal)

            for k in all_m:
                all_m[k].append(compute_metrics(ens_prob, labels)[k])
                unet_m[k].append(compute_metrics(u_prob,  labels)[k])
                dlab_m[k].append(compute_metrics(d_prob,  labels)[k])

            if len(vis_unet_imgs) < NUM_VIS:
                n_take = min(NUM_VIS - len(vis_unet_imgs), u_imgs.size(0))
                vis_unet_imgs.extend(u_imgs[:n_take].cpu())
                vis_labels.extend(labels[:n_take].cpu())
                vis_ens.extend(ens_prob[:n_take].cpu())
                vis_up.extend(u_prob[:n_take].cpu())
                vis_dp.extend(d_prob[:n_take].cpu())
                vis_ids.extend(chip_ids[:n_take])

    # ── Print results ──
    col_w = 12
    print(f"\n{'─'*55}")
    print(f"  {'Metric':<{col_w}} {'UNet':>{col_w}} {'DeepLabV3+':>{col_w}} {'Ensemble':>{col_w}}")
    print(f"{'─'*55}")
    for k in ["accuracy", "precision", "recall", "iou"]:
        u = np.mean(unet_m[k])
        d = np.mean(dlab_m[k])
        e = np.mean(all_m[k])
        best = max(u, d, e)
        # mark best with *
        u_s = f"{u:.4f}{'*' if u==best else ' '}"
        d_s = f"{d:.4f}{'*' if d==best else ' '}"
        e_s = f"{e:.4f}{'*' if e==best else ' '}"
        print(f"  {k.capitalize():<{col_w}} {u_s:>{col_w}} {d_s:>{col_w}} {e_s:>{col_w}}")
    print(f"{'─'*55}")
    print("  (* = best for that metric)\n")

    # ── Visualise ──
    if SAVE_PLOTS:
        save_visualisations(
            torch.stack(vis_unet_imgs),
            torch.stack(vis_labels),
            torch.stack(vis_ens),
            torch.stack(vis_up),
            torch.stack(vis_dp),
            vis_ids
        )

    print("  Done.\n")


if __name__ == "__main__":
    main()


  Ensemble Evaluation: UNet + DeepLabV3+
  Device  : cuda
  Weights : UNet=0.5  DeepLab=0.5

  Usable chips : 11748
  Evaluating on TEST set (2352 chips)

  ✓ UNet loaded
  ✓ DeepLabV3+ loaded



  Evaluating: 100%|██████████████████████████████████████████████████████████████████| 588/588 [02:18<00:00,  4.25it/s]



───────────────────────────────────────────────────────
  Metric               UNet   DeepLabV3+     Ensemble
───────────────────────────────────────────────────────
  Accuracy          0.8853       0.9214*      0.8994 
  Precision         0.9083       0.9346       0.9370*
  Recall            0.9035       0.9291*      0.8989 
  Iou               0.8264       0.8739*      0.8470 
───────────────────────────────────────────────────────
  (* = best for that metric)

  Visualisations saved → ensemble_plots\ensemble_predictions.png
  Done.

